In [1]:
import pandas as pd
enron = pd.read_csv('../data/enron_model_ft.csv')

/var/folders/4y/0t9f134178bbfj5m299wl_7h0000gn/T/ipykernel_28820/2732611547.py:2: DtypeWarning: Columns (2,4,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  enron = pd.read_csv('../data/enron_model_ft.csv')


In [2]:
enron.head()

,Unnamed: 0,EmailSend,EmailReply,SubjectSend,SubjectReply,From,To,DateSend,DateReply,Context,...,syllable_count,avg_word_length,stop_word_ratio,response_time,Response_Within_15_Mins,Response_Within_60_Mins,Response_Within_120_Mins,Response_Within_3_hours,has_question,exclamation_count
0,0,"Nikki, Thanks for the note and I hope your doi...",Hello hello! I was so worried that I had said ...,Portland,Re: Portland,bill.williams@enron.com,-nikole@excite.com,2001-06-18 22:44:00,2001-06-19 15:49:00,NaN,...,717,3.920078,0.547758,1025.0,0,0,0,0,1,1
1,1,Jim Lokay Sales Representative British Parts I...,HI,Call when you receive this (no rush),RE: Call when you receive this (no rush),548@britishparts.com,michelle.lokay@enron.com,2002-03-19 08:30:00,2002-03-19 08:34:00,NaN,...,24,5.538462,0.000000,4.0,1,1,1,1,0,0
2,2,Congratulations! Have a good sleep.,Just woke up...thnx for your note. I believe t...,Thanks and,Re: Thanks and,louise.kitchen@enron.com,8774754543@skytel.com,2002-01-15 12:30:00,2002-01-15 16:29:00,NaN,...,9,5.800000,0.400000,239.0,0,0,0,0,0,1
3,3,Test,Call back : Geir.Solberg@enron.com|Test|Test,Test,Re: Test,geir.solberg@enron.com,8776519147@skytel.com,2002-01-19 10:55:00,2002-01-19 10:56:00,NaN,...,1,4.000000,0.000000,1.0,1,1,1,1,0,0
4,4,We are dropping a lot of marketers. It would b...,As shankman would say 'working ya',Marketers,RE: Marketers,8777865122@skytel.com,louise.kitchen@enron.com,2002-01-21 15:30:00,2002-01-21 15:44:00,NaN,...,20,3.187500,0.500000,14.0,1,1,1,1,0,0


In [3]:
enron.columns

Index(['Unnamed: 0', 'EmailSend', 'EmailReply', 'SubjectSend', 'SubjectReply',
       'From', 'To', 'DateSend', 'DateReply', 'Context',
       'Response_Within_30_Mins', 'HourSent', 'DaySent', 'IsWeekend',
       'Responded', 'EmailSend_Token_Count', 'syllable_count',
       'avg_word_length', 'stop_word_ratio', 'response_time',
       'Response_Within_15_Mins', 'Response_Within_60_Mins',
       'Response_Within_120_Mins', 'Response_Within_3_hours', 'has_question',
       'exclamation_count'],
      dtype='object')

In [4]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))

driver = GraphDatabase.driver(URI, auth=AUTH)

# quick connection test
driver.verify_connectivity()
print("Connected!")

Connected!


In [5]:
def bucket_urgency(minutes):
    if pd.isna(minutes):
        return 'ghosted'
    elif minutes < 5:
        return 'instant'
    elif minutes < 15:
        return 'fast'
    elif minutes < 24*60:
        return 'same_day'
    else:
        return 'next_day'

enron['urgency_bucket'] = enron['response_time'].apply(bucket_urgency)

In [6]:
edge_data = enron.dropna(subset=['response_time']).groupby(['From', 'To']).agg(
    count=('response_time', 'size'),
    urgency=('urgency_bucket', lambda x: x.mode()[0])
).reset_index()

edge_data = edge_data.sort_values('count', ascending=False).head(50)
print(f"{len(edge_data)} relationships ready to load")

50 relationships ready to load


In [7]:
with driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MERGE (a:Person {email: row.From})
        MERGE (b:Person {email: row.To})
        MERGE (a)-[r:EMAILED]->(b)
        SET r.urgency = row.urgency, r.count = row.count
    """, rows=edge_data.to_dict('records'))

print("Loaded into Neo4j")

Loaded into Neo4j


In [8]:
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

g = Neo4jGraphWidget(driver)
g.show_cypher("MATCH (a)-[r]->(b) RETURN a, r, b")

GraphWidget(layout=Layout(height='800px', width='100%'))

In [9]:
enron_valid = enron[enron['response_time'] >= 0]

sender_speed = (
    enron_valid
    .groupby('From')['response_time']
    .agg(avg_response='mean', n_replies='count')
    .query('n_replies >= 5')
    .sort_values('avg_response')
)

fastest_responders = sender_speed.head(20)
fastest_responders

,avg_response,n_replies
From,,
"Maggi, Mike",1.851064,47
ted.evans@enron.com,2.600000,5
"Rybarski, Amanda",4.000000,10
chance.rabon@enron.com,5.200000,5
tom.moran@enron.com,6.200000,5
mjillard@beaconelectric.com,8.714286,7
davis.thames@enron.com,9.666667,6
john.knock@elpaso.com,10.000000,5
llmaser@degeurinrealty.com,10.000000,5


In [10]:
# 1. Compute in-degree in Neo4j and store it on each Person node
with driver.session() as session:
    session.run("""
        MATCH (p:Person)
        OPTIONAL MATCH (p)<-[r:EMAILED]-()
        WITH p, count(r) AS in_degree
        SET p.in_degree = in_degree
    """)

# 2. Flag your fastest responders (from the sender_speed table you already built)
with driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MERGE (p:Person {email: row.From})
        SET p.avg_response = row.avg_response, p.fast_responder = true
    """, rows=fastest_responders.reset_index().to_dict('records'))

In [11]:
g = Neo4jGraphWidget(driver)

g.add_node_configuration(
    "Person",
    color=lambda node: '#e63946' if node['properties'].get('fast_responder') else '#a8dadc',
    size=lambda node: (20 + node['properties'].get('in_degree', 0) * 3, 20 + node['properties'].get('in_degree', 0) * 3),
    text=lambda node: node['properties'].get('email', '').split('@')[0],
)

g.show_cypher("MATCH (a)-[r]->(b) RETURN a, r, b")

GraphWidget(layout=Layout(height='800px', width='100%'))